# Auto Loader - all Bronze tables

Each source has its own landing directory, schema location, checkpoint,
and Bronze target. Unchanged folders become no-ops on later runs.


In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
)

CATALOG = "data_lakehouse_databricks"
BRONZE_SCHEMA = "bronze"

BASE = (
    "/Volumes/data_lakehouse_databricks/"
    "bronze/landing_vol"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{BRONZE_SCHEMA}"
)


def raw_string_schema(*columns):
    return StructType([
        StructField(
            column,
            StringType(),
            True,
        )
        for column in columns
    ])


## Source configuration

In [ ]:
TABLES = {
    "customers": {
        "source": f"{BASE}/landing_cust",
        "schema_location": (
            f"{BASE}/_schemas/landing_cust"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/landing_cust"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_crm_cust_info"
        ),
        "schema": raw_string_schema(
            "cst_id",
            "cst_key",
            "cst_firstname",
            "cst_lastname",
            "cst_marital_status",
            "cst_gndr",
            "cst_create_date",
        ),
    },

    "sales": {
        "source": f"{BASE}/landing_sales",
        "schema_location": (
            f"{BASE}/_schemas/landing_sales"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/landing_sales"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_crm_sales_details"
        ),
        "schema": raw_string_schema(
            "sls_ord_num",
            "sls_prd_key",
            "sls_cust_id",
            "sls_order_dt",
            "sls_ship_dt",
            "sls_due_dt",
            "sls_sales",
            "sls_quantity",
            "sls_price",
        ),
    },

    "product": {
        "source": f"{BASE}/landing_product",
        "schema_location": (
            f"{BASE}/_schemas/landing_product"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/landing_product"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_crm_prd_info"
        ),
        "schema": raw_string_schema(
            "prd_id",
            "prd_key",
            "prd_nm",
            "prd_cost",
            "prd_line",
            "prd_start_dt",
            "prd_end_dt",
        ),
    },

    "customer_demographics": {
        "source": (
            f"{BASE}/"
            "landing_customer_demographics"
        ),
        "schema_location": (
            f"{BASE}/_schemas/"
            "landing_customer_demographics"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/"
            "landing_customer_demographics"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_erp_cust_az12"
        ),
        "schema": raw_string_schema(
            "CID",
            "BDATE",
            "GEN",
        ),
    },

    "customer_location": {
        "source": (
            f"{BASE}/landing_customer_location"
        ),
        "schema_location": (
            f"{BASE}/_schemas/"
            "landing_customer_location"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/"
            "landing_customer_location"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_erp_loc_a101"
        ),
        "schema": raw_string_schema(
            "CID",
            "CNTRY",
        ),
    },

    "product_category": {
        "source": (
            f"{BASE}/landing_product_category"
        ),
        "schema_location": (
            f"{BASE}/_schemas/"
            "landing_product_category"
        ),
        "checkpoint": (
            f"{BASE}/_checkpoints/"
            "landing_product_category"
        ),
        "target": (
            f"{CATALOG}.{BRONZE_SCHEMA}."
            "bronze_erp_px_cat_g1v2"
        ),
        "schema": raw_string_schema(
            "ID",
            "CAT",
            "SUBCAT",
            "MAINTENANCE",
        ),
    },
}


## Generic Auto Loader function

In [ ]:
def run_autoloader(
    source_name,
    config,
):
    expected_columns = [
        field.name
        for field in config["schema"].fields
    ]

    print("\n" + "=" * 90)
    print(f"AUTO LOADER: {source_name}")
    print(f"Source: {config['source']}")
    print(f"Target: {config['target']}")
    print("=" * 90)

    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option(
            "cloudFiles.format",
            "csv",
        )
        .option(
            "cloudFiles.schemaLocation",
            config["schema_location"],
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "none",
        )
        .option(
            "header",
            "true",
        )
        .schema(
            config["schema"]
        )
        .load(
            config["source"]
        )
        .select(
            *expected_columns
        )
    )

    query = (
        stream_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            config["checkpoint"],
        )
        .trigger(
            availableNow=True
        )
        .toTable(
            config["target"]
        )
    )

    query.awaitTermination()

    progress = query.lastProgress

    num_input_rows = (
        int(
            progress.get(
                "numInputRows",
                0,
            )
        )
        if progress
        else 0
    )

    print(
        f"{source_name}: processed "
        f"{num_input_rows} new rows"
    )

    return {
        "source": source_name,
        "target": config["target"],
        "new_rows": num_input_rows,
    }


## Run all six Bronze ingestion streams

`availableNow=True` processes files not yet recorded in each source's
checkpoint and then terminates.


In [ ]:
results = []

for source_name, config in TABLES.items():
    results.append(
        run_autoloader(
            source_name,
            config,
        )
    )

print("\nAuto Loader run completed.")

for result in results:
    print(
        f"{result['source']}: "
        f"{result['new_rows']} new rows -> "
        f"{result['target']}"
    )

dbutils.notebook.exit(
    "OK|"
    + "|".join(
        (
            f"{result['source']}="
            f"{result['new_rows']}"
        )
        for result in results
    )
)
